# Weber Posterior-Marginalized Feature Learning: Flip Set Analysis

Compares **BayesianSPO_Weber_Agent** (MC-marginalized feature gradient) against
BayesianSPO baselines on Shortest Path 8×8.

Weber et al. marginalize over the posterior when computing the feature gradient:
instead of using only θ̂ = A⁻¹b, we sample θ^(s) ~ N(θ̂, A⁻¹) and average the
SPO+ gradient across samples. This makes the gradient uncertainty-aware.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import jax.numpy as jnp
import yaml
import time
from collections import defaultdict

from lets_plot import *
LetsPlot.setup_html()

In [2]:
# Load config
with open('../experiments/configs/experiment.yaml') as f:
    config = yaml.safe_load(f)

config['datasets']['shortest_path_8x8']['context_dim'] = 20

N_ROUNDS = 1000
N_SEEDS = 3
SEEDS = list(range(N_SEEDS))

DATASETS = ['shortest_path_8x8']

from src.solvers.shortest_path import n_edges_grid
grid_size = config['datasets']['shortest_path_8x8']['grid_size']
N_ARMS = n_edges_grid(grid_size)
K_PER_PATH = 2 * (grid_size - 1)
CONTEXT_DIM = config['datasets']['shortest_path_8x8']['context_dim']

print(f'Rounds: {N_ROUNDS}, Seeds: {N_SEEDS}')
print(f'Grid: {grid_size}x{grid_size}, Edges: {N_ARMS}, Path length: {K_PER_PATH}')
print(f'Context dim: {CONTEXT_DIM}')

Rounds: 1000, Seeds: 3
Grid: 8x8, Edges: 112, Path length: 14
Context dim: 20


## Setup: Flip Set & Tracking Utilities

In [3]:
from src.envs import create_env
from src.agents.base import pregenerate_episode
from src.agents.shared_context import (
    BayesianSPOAgent, BayesianMSEAgent, BayesianSPO_Weber_Agent,
)
from src.agents.bayesian_utils import (
    compute_bayesian_predictions,
    compute_ucb_values,
    update_bayesian_stats,
)
from scipy.stats import spearmanr


def compute_per_arm_std(features, A_inv):
    Ax = jnp.einsum('nij,j->ni', A_inv, features)
    variances = jnp.einsum('i,ni->n', features, Ax)
    return np.asarray(jnp.sqrt(jnp.maximum(variances, 0.0)))


def compute_flip_set(pred, true_vals, solver):
    pred_np = np.asarray(pred, dtype=np.float64)
    true_np = np.asarray(true_vals, dtype=np.float64)
    w_true = np.asarray(solver(true_np))
    w_spo = np.asarray(solver(2 * pred_np - true_np))
    flip_mask = np.abs(w_spo - w_true) > 0.5
    return flip_mask, int(flip_mask.sum()), w_true


def run_agent_with_flip_tracking(
    agent, episode, n_rounds, lr=0.01, full_feedback=False,
):
    import src.solver as _solver_module
    _solver_module._GUROBI_ENV = None

    rounds_data = episode['rounds_data']
    minimize = episode['minimize']

    rewards = []
    regrets = []
    cumulative_regret = []
    flip_set_sizes = []
    contested_sigma_sums = []
    contested_sigma_means = []
    contested_errors = []
    decision_correct = []
    contested_rank_corrs = []
    total_regret = 0.0

    for t in range(n_rounds):
        rd = rounds_data[t]
        X = rd['context']
        true_vals = rd['true_values']
        optimal_val = rd['optimal_value']

        h = agent._features(X)
        pred = compute_bayesian_predictions(h, agent.A_inv, agent.b_bayes)
        pred_np = np.asarray(pred)
        sigmas = compute_per_arm_std(h, agent.A_inv)

        flip_mask, flip_size, w_true = compute_flip_set(pred_np, true_vals, agent.solver)
        flip_set_sizes.append(flip_size)

        if flip_size >= 2:
            contested_sigma_sums.append(float(sigmas[flip_mask].sum()))
            contested_sigma_means.append(float(sigmas[flip_mask].mean()))
            contested_errors.append(float(np.mean(np.abs(pred_np[flip_mask] - true_vals[flip_mask]))))
            import warnings
            with np.errstate(invalid='ignore'), warnings.catch_warnings():
                warnings.simplefilter('ignore')
                rho, _ = spearmanr(pred_np[flip_mask], true_vals[flip_mask])
            contested_rank_corrs.append(float(rho) if np.isfinite(rho) else np.nan)
        elif flip_size == 1:
            contested_sigma_sums.append(float(sigmas[flip_mask].sum()))
            contested_sigma_means.append(float(sigmas[flip_mask].mean()))
            contested_errors.append(float(np.abs(pred_np[flip_mask] - true_vals[flip_mask]).item()))
            contested_rank_corrs.append(np.nan)
        else:
            contested_sigma_sums.append(0.0)
            contested_sigma_means.append(0.0)
            contested_errors.append(0.0)
            contested_rank_corrs.append(np.nan)

        w_pred = np.asarray(agent.solver(pred_np))
        decision_correct.append(int(np.allclose(w_pred, w_true, atol=0.5)))

        selection = agent.act(X)
        selection = np.asarray(selection).flatten()

        agent_val = float(np.sum(true_vals * selection))
        if minimize:
            instant_regret = agent_val - optimal_val
        else:
            instant_regret = optimal_val - agent_val

        total_regret += instant_regret
        rewards.append(agent_val)
        regrets.append(instant_regret)
        cumulative_regret.append(total_regret)

        if full_feedback:
            observed = true_vals.copy()
        else:
            observed = np.zeros_like(true_vals)
            mask = selection > 0.5
            observed[mask] = true_vals[mask]

        if isinstance(agent, (BayesianSPOAgent, BayesianSPO_Weber_Agent)):
            prob_vals = rd.get('prob_values')
            agent.update(X, selection, observed, lr, full_feedback=full_feedback,
                         prob_values=prob_vals)
        else:
            agent.update(X, selection, observed, lr, full_feedback=full_feedback)

    return {
        'rewards': rewards,
        'regrets': regrets,
        'cumulative_regret': cumulative_regret,
        'flip_set_sizes': flip_set_sizes,
        'contested_sigma_sums': contested_sigma_sums,
        'contested_sigma_means': contested_sigma_means,
        'contested_errors': contested_errors,
        'decision_correct': decision_correct,
        'contested_rank_corrs': contested_rank_corrs,
        'final_regret': total_regret,
    }

## Agent Variants

In [4]:
def create_agents(env, seed, config, n_rounds):
    defaults = config['agent_defaults']
    hidden_dim = defaults['hidden_dim'] if env.context_dim > 20 else defaults['hidden_dim_small']
    lambda_reg = defaults['lambda_reg']
    minimize = getattr(env, 'minimize', False)
    solver = env.solve
    sign = -1.0 if minimize else 1.0

    spo_common = dict(
        context_dim=env.context_dim, n_arms=env.n_arms,
        weights=env.weights, capacity=env.capacity,
        hidden_dim=hidden_dim, lambda_reg=lambda_reg,
        minimize=minimize, solver=solver, seed=seed,
        mask_grad=True,
    )

    weber_common = dict(**spo_common)

    mse_common = dict(
        context_dim=env.context_dim, n_arms=env.n_arms,
        weights=env.weights, capacity=env.capacity,
        hidden_dim=hidden_dim, lambda_reg=lambda_reg,
        solver=solver, seed=seed,
    )

    agents = {}

    # Baselines
    agents['SPO (beta=1)'] = BayesianSPOAgent(
        **spo_common, beta=sign * 1.0, decay_rounds=n_rounds,
    )
    agents['SPO (EE 5->0.1, 20%)'] = BayesianSPOAgent(
        **spo_common, beta_start=sign * 5.0, beta_end=sign * 0.1,
        decay_rounds=int(0.2 * n_rounds),
    )

    # Weber per-round variants
    agents['Weber (S=5, beta=1)'] = BayesianSPO_Weber_Agent(
        **weber_common, beta=sign * 1.0, decay_rounds=n_rounds,
        n_mc_samples=5,
    )
    agents['Weber (S=5, EE 5->0.1, 20%)'] = BayesianSPO_Weber_Agent(
        **weber_common, beta_start=sign * 5.0, beta_end=sign * 0.1,
        decay_rounds=int(0.2 * n_rounds),
        n_mc_samples=5,
    )
    agents['Weber (S=5, fisher=0.1, beta=1)'] = BayesianSPO_Weber_Agent(
        **weber_common, beta=sign * 1.0, decay_rounds=n_rounds,
        n_mc_samples=5, fisher_beta=0.1,
    )

    # Weber phased
    agents['Weber-Phased (S=5, K=50, beta=1)'] = BayesianSPO_Weber_Agent(
        **weber_common, beta=sign * 1.0, decay_rounds=n_rounds,
        n_mc_samples=5, phased=True, phase_length=50, phase_epochs=5,
    )

    # MSE baseline
    agents['MSE (beta=1)'] = BayesianMSEAgent(
        **mse_common, beta=sign * 1.0, decay_rounds=n_rounds,
    )

    return agents

## Run Evaluation

In [5]:
all_results = {}

for dataset in DATASETS:
    print(f'\n{"="*60}')
    print(f'DATASET: {dataset}')
    print(f'{"="*60}')

    dataset_config = config['datasets'][dataset].copy()
    dataset_config.pop('env_type', None)
    lr = config['agent_defaults']['lr']

    results = defaultdict(list)

    for seed_idx, seed in enumerate(SEEDS):
        print(f'  Seed {seed_idx+1}/{len(SEEDS)} (seed={seed})', flush=True)

        env = create_env(dataset, seed=seed, **dataset_config)
        episode = pregenerate_episode(env, N_ROUNDS, full_feedback=False)

        env_for_agents = create_env(dataset, seed=seed, **dataset_config)
        agents = create_agents(env_for_agents, seed, config, N_ROUNDS)

        for agent_name, agent in agents.items():
            t0 = time.perf_counter()
            result = run_agent_with_flip_tracking(
                agent, episode, N_ROUNDS, lr=lr, full_feedback=False,
            )
            wall_time = time.perf_counter() - t0
            results[agent_name].append(result)

            final_flip = np.mean(result['flip_set_sizes'][-50:])
            print(f'    {agent_name:45s} regret={result["final_regret"]:8.1f}  '
                  f'flip_final={final_flip:.1f}  ({wall_time:.1f}s)', flush=True)

    all_results[dataset] = dict(results)


DATASET: shortest_path_8x8
  Seed 1/3 (seed=0)
    SPO (beta=1)                                  regret=  3578.7  flip_final=24.9  (2.9s)
    SPO (EE 5->0.1, 20%)                          regret=  3504.7  flip_final=24.8  (2.3s)
    Weber (S=5, beta=1)                           regret=  3745.7  flip_final=25.2  (3.6s)
    Weber (S=5, EE 5->0.1, 20%)                   regret=  3679.1  flip_final=24.8  (3.3s)
    Weber (S=5, fisher=0.1, beta=1)               regret=  3810.6  flip_final=25.4  (3.4s)
    Weber-Phased (S=5, K=50, beta=1)              regret=  4947.4  flip_final=24.5  (13.8s)
    MSE (beta=1)                                  regret=  3772.4  flip_final=24.5  (1.5s)
  Seed 2/3 (seed=1)
    SPO (beta=1)                                  regret=  3441.1  flip_final=22.5  (1.7s)
    SPO (EE 5->0.1, 20%)                          regret=  3255.1  flip_final=22.4  (1.7s)
    Weber (S=5, beta=1)                           regret=  3585.4  flip_final=23.6  (3.3s)
    Weber (S=5, EE 5-

## 1. Summary Table (Last 100 Rounds)

In [6]:
for dataset in DATASETS:
    print(f'\n--- {dataset} ---')
    rows = []
    for agent_name, res_list in all_results[dataset].items():
        final_regrets = [r['final_regret'] for r in res_list]
        mean_flip_last = [np.mean(r['flip_set_sizes'][-100:]) for r in res_list]
        mean_err = [np.mean(r['contested_errors'][-100:]) for r in res_list]
        mean_dec = [np.mean(r['decision_correct'][-100:]) for r in res_list]
        rank_vals = []
        for r in res_list:
            rc = np.array(r['contested_rank_corrs'][-100:])
            rc = rc[~np.isnan(rc)]
            rank_vals.append(np.mean(rc) if len(rc) > 0 else np.nan)

        rows.append({
            'Agent': agent_name,
            'Regret': f'{np.mean(final_regrets):.0f} +/- {np.std(final_regrets):.0f}',
            'Flip': f'{np.mean(mean_flip_last):.1f}',
            'Err': f'{np.mean(mean_err):.3f}',
            'Dec': f'{np.mean(mean_dec):.1%}',
            'Rank': f'{np.nanmean(rank_vals):.3f}',
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print('  Err=contested MAE | Dec=decision acc | Rank=Spearman on flip set (last 100)')


--- shortest_path_8x8 ---
                           Agent       Regret Flip   Err  Dec  Rank
                    SPO (beta=1)  3532 +/- 64 24.3 0.587 2.7% 0.547
            SPO (EE 5->0.1, 20%) 3374 +/- 102 24.3 0.581 2.3% 0.572
             Weber (S=5, beta=1)  3613 +/- 99 24.6 0.596 3.0% 0.591
     Weber (S=5, EE 5->0.1, 20%) 3468 +/- 151 24.6 0.605 4.0% 0.561
 Weber (S=5, fisher=0.1, beta=1) 3628 +/- 130 24.7 0.602 1.7% 0.568
Weber-Phased (S=5, K=50, beta=1)  5000 +/- 37 23.7 0.839 0.0% 0.077
                    MSE (beta=1)  3726 +/- 62 24.8 0.646 3.0% 0.555
  Err=contested MAE | Dec=decision acc | Rank=Spearman on flip set (last 100)


## 2. Cumulative Regret Over Time

In [7]:
def build_regret_df(results, dataset):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            traj = np.array(res['cumulative_regret'])
            step = max(1, len(traj) // 200)
            for t in range(0, len(traj), step):
                rows.append({'Round': t, 'Cumulative Regret': traj[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    reg_df = build_regret_df(all_results[dataset], dataset)
    agg = reg_df.groupby(['Round', 'Agent'])['Cumulative Regret'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Cumulative Regret', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Cumulative Regret: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 3. Flip Set Size Over Time

In [8]:
def build_flip_df(results, dataset):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            flips = np.array(res['flip_set_sizes'])
            step = max(1, len(flips) // 200)
            for t in range(0, len(flips), step):
                rows.append({'Round': t, 'Flip Set Size': flips[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    flip_df = build_flip_df(all_results[dataset], dataset)
    agg = flip_df.groupby(['Round', 'Agent'])['Flip Set Size'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Flip Set Size', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Flip Set Size: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 4. Contested-Arm Uncertainty Over Time

In [9]:
def build_sigma_df(results, dataset, key='contested_sigma_sums', label='Contested Sigma Sum'):
    rows = []
    for agent_name, res_list in results.items():
        for seed_idx, res in enumerate(res_list):
            vals = np.array(res[key])
            step = max(1, len(vals) // 200)
            for t in range(0, len(vals), step):
                rows.append({'Round': t, label: vals[t],
                             'Agent': agent_name, 'Seed': seed_idx, 'Dataset': dataset})
    return pd.DataFrame(rows)

for dataset in DATASETS:
    sigma_df = build_sigma_df(all_results[dataset], dataset)
    agg = sigma_df.groupby(['Round', 'Agent'])['Contested Sigma Sum'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Contested Sigma Sum', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm Uncertainty: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 5. Contested Error + Rank Correlation

In [10]:
for dataset in DATASETS:
    # Contested error
    err_df = build_sigma_df(all_results[dataset], dataset,
                            key='contested_errors', label='Contested MAE')
    agg = err_df.groupby(['Round', 'Agent'])['Contested MAE'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Contested MAE', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm MAE: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

    # Rank correlation
    rank_df = build_sigma_df(all_results[dataset], dataset,
                             key='contested_rank_corrs', label='Rank Corr')
    agg = rank_df.groupby(['Round', 'Agent'])['Rank Corr'].mean().reset_index()
    p = (ggplot(agg, aes('Round', 'Rank Corr', color='Agent'))
         + geom_line(size=1)
         + ggtitle(f'Contested-Arm Rank Correlation: {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 6. Regret vs Flip Set Scatter

In [11]:
for dataset in DATASETS:
    rows = []
    for agent_name, res_list in all_results[dataset].items():
        for res in res_list:
            regrets = np.array(res['regrets'])
            flips = np.array(res['flip_set_sizes'])
            # Smooth with rolling window
            w = 50
            for t in range(w, len(regrets)):
                rows.append({
                    'Mean Regret': np.mean(regrets[t-w:t]),
                    'Mean Flip Size': np.mean(flips[t-w:t]),
                    'Agent': agent_name,
                })
    scat_df = pd.DataFrame(rows)
    p = (ggplot(scat_df, aes('Mean Flip Size', 'Mean Regret', color='Agent'))
         + geom_point(alpha=0.3, size=1)
         + ggtitle(f'Regret vs Flip Set Size (smoothed): {dataset}')
         + theme(legend_position='bottom'))
    p.show()

## 7. T₀ Collapse Time

In [12]:
def estimate_T0(flip_sizes, n_arms, threshold_frac=0.1, window=50):
    threshold = threshold_frac * n_arms
    flips = np.array(flip_sizes)
    for t in range(window, len(flips)):
        if np.mean(flips[t-window:t]) <= threshold:
            return t - window
    return len(flips)  # never collapsed

for dataset in DATASETS:
    print(f'\n--- {dataset}: T_0 estimates ---')
    for agent_name, res_list in all_results[dataset].items():
        t0s = [estimate_T0(r['flip_set_sizes'], N_ARMS) for r in res_list]
        print(f'  {agent_name:45s}  T_0 = {np.mean(t0s):.0f} +/- {np.std(t0s):.0f}')


--- shortest_path_8x8: T_0 estimates ---
  SPO (beta=1)                                   T_0 = 1000 +/- 0
  SPO (EE 5->0.1, 20%)                           T_0 = 1000 +/- 0
  Weber (S=5, beta=1)                            T_0 = 1000 +/- 0
  Weber (S=5, EE 5->0.1, 20%)                    T_0 = 1000 +/- 0
  Weber (S=5, fisher=0.1, beta=1)                T_0 = 1000 +/- 0
  Weber-Phased (S=5, K=50, beta=1)               T_0 = 1000 +/- 0
  MSE (beta=1)                                   T_0 = 1000 +/- 0


## 8. LR Sensitivity: Weber vs Baseline

In [13]:
LR_VALUES = [0.001, 0.01, 0.05]
LR_N_ROUNDS = 1000
LR_N_SEEDS = 3

lr_results = {}  # lr -> agent_name -> list of results

for feat_lr in LR_VALUES:
    print(f'\n--- LR = {feat_lr} ---')
    results = defaultdict(list)
    dataset = DATASETS[0]
    dataset_config = config['datasets'][dataset].copy()
    dataset_config.pop('env_type', None)

    for seed in range(LR_N_SEEDS):
        env = create_env(dataset, seed=seed, **dataset_config)
        episode = pregenerate_episode(env, LR_N_ROUNDS, full_feedback=False)

        env2 = create_env(dataset, seed=seed, **dataset_config)
        minimize = getattr(env2, 'minimize', False)
        sign = -1.0 if minimize else 1.0
        defaults = config['agent_defaults']
        hidden_dim = defaults['hidden_dim'] if env2.context_dim > 20 else defaults['hidden_dim_small']

        common = dict(
            context_dim=env2.context_dim, n_arms=env2.n_arms,
            weights=env2.weights, capacity=env2.capacity,
            hidden_dim=hidden_dim, lambda_reg=defaults['lambda_reg'],
            minimize=minimize, solver=env2.solve, seed=seed,
            mask_grad=True, lr=feat_lr,
        )

        agents = {
            f'SPO (lr={feat_lr})': BayesianSPOAgent(
                **common, beta=sign * 1.0, decay_rounds=LR_N_ROUNDS),
            f'Weber (lr={feat_lr})': BayesianSPO_Weber_Agent(
                **common, beta=sign * 1.0, decay_rounds=LR_N_ROUNDS, n_mc_samples=5),
            f'Weber+Fisher (lr={feat_lr})': BayesianSPO_Weber_Agent(
                **common, beta=sign * 1.0, decay_rounds=LR_N_ROUNDS,
                n_mc_samples=5, fisher_beta=0.1),
        }

        for agent_name, agent in agents.items():
            t0 = time.perf_counter()
            result = run_agent_with_flip_tracking(
                agent, episode, LR_N_ROUNDS, lr=feat_lr, full_feedback=False)
            wall_time = time.perf_counter() - t0
            results[agent_name].append(result)
            print(f'  {agent_name:40s} regret={result["final_regret"]:8.1f} ({wall_time:.1f}s)')

    lr_results[feat_lr] = dict(results)

# Summary table
print('\n--- LR Sensitivity Summary ---')
rows = []
for feat_lr in LR_VALUES:
    for agent_name, res_list in lr_results[feat_lr].items():
        final_regrets = [r['final_regret'] for r in res_list]
        mean_flip = [np.mean(r['flip_set_sizes'][-100:]) for r in res_list]
        rows.append({
            'LR': feat_lr,
            'Agent': agent_name,
            'Regret': f'{np.mean(final_regrets):.0f} +/- {np.std(final_regrets):.0f}',
            'Flip': f'{np.mean(mean_flip):.1f}',
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))


--- LR = 0.001 ---
  SPO (lr=0.001)                           regret=  3578.7 (1.6s)
  Weber (lr=0.001)                         regret=  3745.7 (3.3s)
  Weber+Fisher (lr=0.001)                  regret=  3810.6 (3.3s)
  SPO (lr=0.001)                           regret=  3441.1 (1.7s)
  Weber (lr=0.001)                         regret=  3585.4 (3.2s)
  Weber+Fisher (lr=0.001)                  regret=  3558.5 (3.3s)
  SPO (lr=0.001)                           regret=  3576.0 (1.5s)
  Weber (lr=0.001)                         regret=  3507.8 (3.3s)
  Weber+Fisher (lr=0.001)                  regret=  3515.0 (3.3s)

--- LR = 0.01 ---
  SPO (lr=0.01)                            regret=  2615.7 (1.6s)
  Weber (lr=0.01)                          regret=  3040.1 (3.3s)
  Weber+Fisher (lr=0.01)                   regret=  3014.2 (3.2s)
  SPO (lr=0.01)                            regret=  2972.5 (1.6s)
  Weber (lr=0.01)                          regret=  3037.5 (3.4s)
  Weber+Fisher (lr=0.01)             

## 9. Scaling: 8×8, 12×12, 16×16

In [14]:
from src.solvers.shortest_path import n_edges_grid, ShortestPathSolver

GRID_SIZES = [8, 12, 16]
SCALE_N_ROUNDS = 500
SCALE_N_SEEDS = 2

scaling_results = {}  # grid_size -> agent_name -> list of results

for gs in GRID_SIZES:
    print(f'\n{"="*60}')
    print(f'Grid: {gs}x{gs}, Edges: {n_edges_grid(gs)}')
    print(f'{"="*60}')

    gs_config = {
        'grid_size': gs,
        'context_dim': 20,
        'noise_width': 0.25,
    }

    results = defaultdict(list)

    for seed in range(SCALE_N_SEEDS):
        env = create_env('shortest_path', seed=seed, **gs_config)
        episode = pregenerate_episode(env, SCALE_N_ROUNDS, full_feedback=False)

        env2 = create_env('shortest_path', seed=seed, **gs_config)
        minimize = getattr(env2, 'minimize', False)
        sign = -1.0 if minimize else 1.0
        defaults = config['agent_defaults']
        hidden_dim = defaults['hidden_dim'] if env2.context_dim > 20 else defaults['hidden_dim_small']

        common = dict(
            context_dim=env2.context_dim, n_arms=env2.n_arms,
            weights=env2.weights, capacity=env2.capacity,
            hidden_dim=hidden_dim, lambda_reg=defaults['lambda_reg'],
            minimize=minimize, solver=env2.solve, seed=seed,
            mask_grad=True,
        )

        agents = {
            'SPO (beta=1)': BayesianSPOAgent(
                **common, beta=sign * 1.0, decay_rounds=SCALE_N_ROUNDS),
            'Weber (S=5, beta=1)': BayesianSPO_Weber_Agent(
                **common, beta=sign * 1.0, decay_rounds=SCALE_N_ROUNDS, n_mc_samples=5),
            'Weber+Fisher (S=5, beta=1)': BayesianSPO_Weber_Agent(
                **common, beta=sign * 1.0, decay_rounds=SCALE_N_ROUNDS,
                n_mc_samples=5, fisher_beta=0.1),
            'MSE (beta=1)': BayesianMSEAgent(
                context_dim=env2.context_dim, n_arms=env2.n_arms,
                weights=env2.weights, capacity=env2.capacity,
                hidden_dim=hidden_dim, lambda_reg=defaults['lambda_reg'],
                solver=env2.solve, seed=seed,
                beta=sign * 1.0, decay_rounds=SCALE_N_ROUNDS),
        }

        lr = defaults['lr']
        for agent_name, agent in agents.items():
            t0 = time.perf_counter()
            result = run_agent_with_flip_tracking(
                agent, episode, SCALE_N_ROUNDS, lr=lr, full_feedback=False)
            wall_time = time.perf_counter() - t0
            results[agent_name].append(result)
            print(f'  {agent_name:40s} regret={result["final_regret"]:8.1f} ({wall_time:.1f}s)')

    scaling_results[gs] = dict(results)

# Scaling summary
print('\n--- Scaling Summary ---')
rows = []
for gs in GRID_SIZES:
    for agent_name, res_list in scaling_results[gs].items():
        final_regrets = [r['final_regret'] for r in res_list]
        mean_flip = [np.mean(r['flip_set_sizes'][-100:]) for r in res_list]
        rows.append({
            'Grid': f'{gs}x{gs}',
            'Edges': n_edges_grid(gs),
            'Agent': agent_name,
            'Regret': f'{np.mean(final_regrets):.0f} +/- {np.std(final_regrets):.0f}',
            'Flip': f'{np.mean(mean_flip):.1f}',
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))


Grid: 8x8, Edges: 112
  SPO (beta=1)                             regret=  2134.1 (0.8s)
  Weber (S=5, beta=1)                      regret=  2199.0 (1.6s)
  Weber+Fisher (S=5, beta=1)               regret=  2188.2 (1.7s)
  MSE (beta=1)                             regret=  2111.4 (0.8s)
  SPO (beta=1)                             regret=  2067.4 (0.9s)
  Weber (S=5, beta=1)                      regret=  2094.1 (1.8s)
  Weber+Fisher (S=5, beta=1)               regret=  2121.1 (1.6s)
  MSE (beta=1)                             regret=  2145.1 (0.8s)

Grid: 12x12, Edges: 264
  SPO (beta=1)                             regret=  3727.5 (1.4s)
  Weber (S=5, beta=1)                      regret=  3550.9 (2.5s)
  Weber+Fisher (S=5, beta=1)               regret=  3762.4 (2.4s)
  MSE (beta=1)                             regret=  3758.8 (0.8s)
  SPO (beta=1)                             regret=  3514.6 (1.0s)
  Weber (S=5, beta=1)                      regret=  3442.8 (2.2s)
  Weber+Fisher (S=5, beta=1)